In [0]:
bronze_path   = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/bronze/'
silver_path   = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/silver/'
gold_path     = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/gold/'
resource_path = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/resource/'

In [0]:

# criar várias tabelas temporárias de forma prática
bronze_map = {
    "tmp_bronze_brands":     f"{bronze_path}/brands/",
    "tmp_bronze_categories": f"{bronze_path}/categories/",
    "tmp_bronze_customers":  f"{bronze_path}/customers/",
    "tmp_bronze_order_items":f"{bronze_path}/order_items/",
    "tmp_bronze_orders":     f"{bronze_path}/orders/",
    "tmp_bronze_products":   f"{bronze_path}/products/",
    "tmp_bronze_staffs":     f"{bronze_path}/staffs/",
    "tmp_bronze_stocks":     f"{bronze_path}/stocks/",
    "tmp_bronze_stores":     f"{bronze_path}/stores/",
}

for view_name, path in bronze_map.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)
    )


In [0]:
df_customers_silver = spark.sql("""
select
  customer_id,
  first_name,
  last_name,
  phone,
  email,
  street,
  city,
  state,
  zip_code
from
  tmp_bronze_customers
where
  1 = 1
  and phone is not null
  and phone not in ('NULL', 'NULL ')
  and email is not null
  and email not in ('NULL', 'NULL ')

""" )


#salvar em parquet como delta tabela de teste SILVER
df_customers_silver.write\
    .mode('overwrite')\
    .format('delta')\
    .option('mergeSchema', 'true')\
    .save(f'{silver_path}/customers')